##Schema of Dataset

AcresBurned - Acres of land affected by wildfires

Active - If the fire is active or contained?

ArchiveYear - Year the incident took place

CalFireIncident - Is the incident treated as a CalFire incident?

Counties - County name

CountyIds - County id

Latitude - Latitude of the Wildfire incident

Location - Description of the location

Longitude - Longitude of the Wildfire incident

Name - Name of the fire incident

Status - Status of fire finalized or inactive

####Reading california forest fire dataset from google drive

In [ ]:
from google.colab import drive
# NOTE: this will pop up asking for google login permission1
drive.mount('/content/drive')
# linux command to list the files under linux running
#Colab Jupyter notebook (prints dir/files/links in your Drive)
!ls -ltr /content/drive/MyDrive/ | grep *.csv

Mounted at /content/drive
ls: '/content/drive/MyDrive/dataset (1)': No such file or directory
ls: /content/drive/MyDrive/dataset_csv: No such file or directory
ls: /content/drive/MyDrive/dataset: No such file or directory


In [ ]:
!ls -ltr /content/drive/MyDrive/surya_data/California_Fire_Incidents_V1.csv

-rw------- 1 root root 203166 Jan  5  2023 /content/drive/MyDrive/surya_data/California_Fire_Incidents_V1.csv


In [ ]:
import csv
import re
import random
import traceback
from tabulate import tabulate
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import xlrd
import glob
import numpy as np
import datetime
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.neural_network import MLPClassifier,MLPRegressor
from sklearn.tree import DecisionTreeRegressor as dtr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import r2_score
from sklearn import svm
from sklearn.svm import SVC
from sklearn import metrics
from sklearn import preprocessing
from sklearn import linear_model
from scipy import stats
import warnings
from sklearn.metrics import confusion_matrix
warnings.filterwarnings(action='ignore')
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
def pretty_print(df, n):
  print(tabulate(df.head(n),headers='keys',tablefmt='psql'))

In [ ]:
df_california=pd.read_csv('/content/drive/MyDrive/surya_data/California_Fire_Incidents_V1.csv')
pretty_print(df_california,1)
print('Shape', df_california.shape)

+----+---------------+----------+---------------+-------------------+------------+-------------+------------+-----------------------------------------+-------------+----------+-----------+
|    |   AcresBurned | Active   |   ArchiveYear | CalFireIncident   | Counties   |   CountyIds |   Latitude | Location                                |   Longitude | Name     | Status    |
|----+---------------+----------+---------------+-------------------+------------+-------------+------------+-----------------------------------------+-------------+----------+-----------|
|  0 |        257314 | False    |          2013 | True              | Tuolumne   |          55 |     37.857 | 3 miles east of Groveland along Hwy 120 |    -120.086 | Rim Fire | Finalized |
+----+---------------+----------+---------------+-------------------+------------+-------------+------------+-----------------------------------------+-------------+----------+-----------+
Shape (1636, 11)


####Dtypes

In [ ]:
df_california.info()
df_california.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1636 entries, 0 to 1635
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   AcresBurned      1633 non-null   float64
 1   Active           1636 non-null   bool   
 2   ArchiveYear      1636 non-null   int64  
 3   CalFireIncident  1636 non-null   bool   
 4   Counties         1636 non-null   object 
 5   CountyIds        1636 non-null   object 
 6   Latitude         1636 non-null   float64
 7   Location         1636 non-null   object 
 8   Longitude        1636 non-null   float64
 9   Name             1636 non-null   object 
 10  Status           1636 non-null   object 
dtypes: bool(2), float64(3), int64(1), object(5)
memory usage: 118.4+ KB


AcresBurned        float64
Active                bool
ArchiveYear          int64
CalFireIncident       bool
Counties            object
CountyIds           object
Latitude           float64
Location            object
Longitude          float64
Name                object
Status              object
dtype: object

####Data Statistics

In [ ]:
df_california.describe().T

,count,mean,std,min,25%,50%,75%,max
AcresBurned,1633.0,4589.443968,27266.337722,0.00000,35.000000,100.000000,422.000000,410203.0000
ArchiveYear,1636.0,2016.608802,1.845340,2013.00000,2015.000000,2017.000000,2018.000000,2019.0000
Latitude,1636.0,37.203975,135.401380,-120.25800,34.165891,37.104065,39.086808,5487.0000
Longitude,1636.0,-108.082642,37.006927,-124.19629,-121.768358,-120.461560,-117.474073,118.9082


####Drop columns

In [ ]:
df_california_v1 = df_california.drop(columns=['Active', 'Name','Status','Location'])
pretty_print(df_california_v1,1)
print("Shape:", df_california_v1.shape)

+----+---------------+---------------+-------------------+------------+-------------+------------+-------------+
|    |   AcresBurned |   ArchiveYear | CalFireIncident   | Counties   |   CountyIds |   Latitude |   Longitude |
|----+---------------+---------------+-------------------+------------+-------------+------------+-------------|
|  0 |        257314 |          2013 | True              | Tuolumne   |          55 |     37.857 |    -120.086 |
+----+---------------+---------------+-------------------+------------+-------------+------------+-------------+
Shape: (1636, 7)


###Preprocessing Data

#####Text data included with the numeric data(Counties). So we need to encode that in some numeric form before splitting the train test data

In [ ]:
df_explode_countyids = df_california_v1.copy()
pretty_print(df_explode_countyids,1)

+----+---------------+---------------+-------------------+------------+-------------+------------+-------------+
|    |   AcresBurned |   ArchiveYear | CalFireIncident   | Counties   |   CountyIds |   Latitude |   Longitude |
|----+---------------+---------------+-------------------+------------+-------------+------------+-------------|
|  0 |        257314 |          2013 | True              | Tuolumne   |          55 |     37.857 |    -120.086 |
+----+---------------+---------------+-------------------+------------+-------------+------------+-------------+


In [ ]:
# Ordinalencoding on multiple columns conversion of categorical to numeric values without labels
enc = OrdinalEncoder()
enc.fit(df_explode_countyids[["CalFireIncident","Counties", "CountyIds"]])
df_explode_countyids[["CalFireIncident","Counties", "CountyIds"]] = enc.transform(df_explode_countyids[["CalFireIncident","Counties", "CountyIds"]])
pretty_print(df_explode_countyids, 10)


OrdinalEncoder()

+----+---------------+---------------+-------------------+------------+-------------+------------+-------------+
|    |   AcresBurned |   ArchiveYear |   CalFireIncident |   Counties |   CountyIds |   Latitude |   Longitude |
|----+---------------+---------------+-------------------+------------+-------------+------------+-------------|
|  0 |        257314 |          2013 |                 1 |         55 |          63 |    37.857  |    -120.086 |
|  1 |         30274 |          2013 |                 1 |         17 |          13 |    34.5856 |    -118.423 |
|  2 |         27531 |          2013 |                 1 |         32 |          36 |    33.7095 |    -116.729 |
|  3 |         27440 |          2013 |                 0 |         30 |          34 |    39.12   |    -120.65  |
|  4 |         24251 |          2013 |                 1 |         56 |          64 |     0      |       0     |
|  5 |         22992 |          2013 |                 0 |          9 |           2 |    37.279 

In [ ]:
#Label Encoding , is used only once on single column of dataframe, to use particular column as label.
label_encoder = LabelEncoder()

df_explode_countyids ['AcresBurned'] = label_encoder.fit_transform(df_explode_countyids['AcresBurned'])
pretty_print(df_explode_countyids,2)

+----+---------------+---------------+-------------------+------------+-------------+------------+-------------+
|    |   AcresBurned |   ArchiveYear |   CalFireIncident |   Counties |   CountyIds |   Latitude |   Longitude |
|----+---------------+---------------+-------------------+------------+-------------+------------+-------------|
|  0 |           631 |          2013 |                 1 |         55 |          63 |    37.857  |    -120.086 |
|  1 |           588 |          2013 |                 1 |         17 |          13 |    34.5856 |    -118.423 |
+----+---------------+---------------+-------------------+------------+-------------+------------+-------------+


In [ ]:
def convert_list(row):
  mList = [int(e) if e.isdigit() else e for e in str(row['CountyIds']).split(',')]
  return mList

#Create new column to store value to str or int by passing function on dataframe
df_explode_countyids['CountyIds_new'] = df_explode_countyids.apply(convert_list, axis=1)
df_explode_countyids = df_explode_countyids.explode('CountyIds_new')
#pretty_print(df_explode_countyids,2)
df_explode_countyids.dtypes

AcresBurned          int64
ArchiveYear          int64
CalFireIncident    float64
Counties           float64
CountyIds          float64
Latitude           float64
Longitude          float64
CountyIds_new       object
dtype: object

In [ ]:
def fill_na_0(row):
  if str(row['CountyIds_new']).isnumeric():
   return int(row['CountyIds_new'])
  else:
   return 0

df_explode_countyids['County_new_Ids'] = df_explode_countyids.apply(fill_na_0, axis=1)
pretty_print(df_explode_countyids,1)
df_explode_countyids.dtypes

+----+---------------+---------------+-------------------+------------+-------------+------------+-------------+-----------------+------------------+
|    |   AcresBurned |   ArchiveYear |   CalFireIncident |   Counties |   CountyIds |   Latitude |   Longitude |   CountyIds_new |   County_new_Ids |
|----+---------------+---------------+-------------------+------------+-------------+------------+-------------+-----------------+------------------|
|  0 |           631 |          2013 |                 1 |         55 |          63 |     37.857 |    -120.086 |              63 |                0 |
+----+---------------+---------------+-------------------+------------+-------------+------------+-------------+-----------------+------------------+


AcresBurned          int64
ArchiveYear          int64
CalFireIncident    float64
Counties           float64
CountyIds          float64
Latitude           float64
Longitude          float64
CountyIds_new       object
County_new_Ids       int64
dtype: object

In [ ]:
df_explode_countyids = df_explode_countyids.drop(columns=['CountyIds','CountyIds_new'])
# pretty_print(df_explode_countyids,2)
df_explode_countyids.dtypes

AcresBurned          int64
ArchiveYear          int64
CalFireIncident    float64
Counties           float64
Latitude           float64
Longitude          float64
County_new_Ids       int64
dtype: object

In [ ]:
#created new function for lambda
m = df_explode_countyids['AcresBurned'].mean()
print("mean",m)

sd = df_explode_countyids['AcresBurned'].std()
print("standard deviation",sd)

def lambda_dup(df_explode_countyids):
  if (df_explode_countyids['AcresBurned'] == 0):
        return 0
  elif (df_explode_countyids['AcresBurned'] <= m + 1*sd):
        return 1
  elif (df_explode_countyids['AcresBurned'] <= m+ 2*sd):
        return 2
  elif (df_explode_countyids['AcresBurned'] <= m + 3*sd):
        return 3
  else:
        return 0

mean 172.56112469437653
standard deviation 182.75698824062303


In [ ]:
df_explode_countyids['lambda_dup_num'] = df_explode_countyids.apply(lambda_dup, axis=1)
pretty_print(df_explode_countyids,1)
df_explode_countyids.dtypes

+----+---------------+---------------+-------------------+------------+------------+-------------+------------------+------------------+
|    |   AcresBurned |   ArchiveYear |   CalFireIncident |   Counties |   Latitude |   Longitude |   County_new_Ids |   lambda_dup_num |
|----+---------------+---------------+-------------------+------------+------------+-------------+------------------+------------------|
|  0 |           631 |          2013 |                 1 |         55 |     37.857 |    -120.086 |                0 |                3 |
+----+---------------+---------------+-------------------+------------+------------+-------------+------------------+------------------+


AcresBurned          int64
ArchiveYear          int64
CalFireIncident    float64
Counties           float64
Latitude           float64
Longitude          float64
County_new_Ids       int64
lambda_dup_num       int64
dtype: object

In [ ]:
def ordinal_encoding(df_explode_countyids,column,ordering):
  df_explode_countyids = df_explode_countyids.copy()
  df_explode_countyids[column] = df_explode_countyids[column].apply(lambda_dup)
  return df_explode_countyids

In [ ]:
def preprocessing(df_explode_countyids,task):
  df_explode_countyids=df_explode_countyids.copy()

  if task=='Regression':
    Y=df_explode_countyids['lambda_dup_num']
  elif task=='Classification':
    Y=df_explode_countyids['lambda_dup_num']

  X=df_explode_countyids.drop(['lambda_dup_num','AcresBurned'],axis=1)

  X_train,X_test,Y_train,Y_test=train_test_split(X,Y,train_size=0.65,shuffle=True,random_state=1)

  scaler=StandardScaler()
  scaler.fit(X_train)

  X_train=pd.DataFrame(scaler.transform(X_train),columns=X.columns)
  X_test=pd.DataFrame(scaler.transform(X_test),columns=X.columns)
  return X_train,X_test,Y_train,Y_test

####Splitting and Testing models

In [ ]:
X_train, X_test, Y_train, Y_test = preprocessing(df_explode_countyids, task='Classification')
X_train.head(1)
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)

,ArchiveYear,CalFireIncident,Counties,Latitude,Longitude,County_new_Ids
0,1.292532,0.560328,0.108824,0.373163,-0.351624,0.0


(1063, 6) (573, 6) (1063,) (573,)


In [ ]:
df_explode_countyids["lambda_dup_num"].value_counts()

1    1294
2     202
3     112
0      28
Name: lambda_dup_num, dtype: int64

In [ ]:
dfc = df_explode_countyids.copy()
#print(dfc)

Classifiers

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.metrics  import f1_score,accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_validate
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score
from sklearn.utils import class_weight
from sklearn import naive_bayes
from sklearn import metrics

####Best working C parameter values

In [ ]:
svm = SVC(C=0.2, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=0.2, class_weight='balanced', gamma='auto', kernel='linear',
    max_iter=1000)

{'fit_time': array([0.03642273, 0.03903508]), 'score_time': array([0.01525235, 0.01532698]), 'test_score': array([0.66541353, 0.54425612])}
               precision    recall  f1-score   support

      no_fire     0.0763    0.8333    0.1399        12
     low_fire     0.8416    0.8397    0.8407       443
moderate_fire     0.0000    0.0000    0.0000        77
    high_fire     0.0000    0.0000    0.0000        41

     accuracy                         0.6667       573
    macro avg     0.2295    0.4183    0.2451       573
 weighted avg     0.6523    0.6667    0.6529       573

0.4182656132430399


In [ ]:
svm = SVC(C=0.3, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=0.3, class_weight='balanced', gamma='auto', kernel='linear',
    max_iter=1000)

{'fit_time': array([0.01993799, 0.02580261]), 'score_time': array([0.00935078, 0.01692057]), 'test_score': array([0.66541353, 0.54425612])}
               precision    recall  f1-score   support

      no_fire     0.0763    0.8333    0.1399        12
     low_fire     0.8416    0.8397    0.8407       443
moderate_fire     0.0000    0.0000    0.0000        77
    high_fire     0.0000    0.0000    0.0000        41

     accuracy                         0.6667       573
    macro avg     0.2295    0.4183    0.2451       573
 weighted avg     0.6523    0.6667    0.6529       573

0.4182656132430399


In [ ]:
svm = SVC(C=0.8, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=0.8, class_weight='balanced', gamma='auto', kernel='linear',
    max_iter=1000)

{'fit_time': array([0.03546643, 0.04087234]), 'score_time': array([0.01581883, 0.01645303]), 'test_score': array([0.66541353, 0.54613936])}
               precision    recall  f1-score   support

      no_fire     0.0763    0.8333    0.1399        12
     low_fire     0.8416    0.8397    0.8407       443
moderate_fire     0.0000    0.0000    0.0000        77
    high_fire     0.0000    0.0000    0.0000        41

     accuracy                         0.6667       573
    macro avg     0.2295    0.4183    0.2451       573
 weighted avg     0.6523    0.6667    0.6529       573

0.4182656132430399


In [ ]:
svm = SVC(C=1, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=1, class_weight='balanced', gamma='auto', kernel='linear', max_iter=1000)

{'fit_time': array([0.02013397, 0.02153301]), 'score_time': array([0.00881529, 0.00935173]), 'test_score': array([0.66541353, 0.54990584])}
               precision    recall  f1-score   support

      no_fire     0.0763    0.8333    0.1399        12
     low_fire     0.8416    0.8397    0.8407       443
moderate_fire     0.0000    0.0000    0.0000        77
    high_fire     0.0000    0.0000    0.0000        41

     accuracy                         0.6667       573
    macro avg     0.2295    0.4183    0.2451       573
 weighted avg     0.6523    0.6667    0.6529       573

0.4182656132430399


In [ ]:
svm = SVC(C=2, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=2, class_weight='balanced', gamma='auto', kernel='linear', max_iter=1000)

{'fit_time': array([0.02029896, 0.02758455]), 'score_time': array([0.01010513, 0.0091393 ]), 'test_score': array([0.66729323, 0.54425612])}
               precision    recall  f1-score   support

      no_fire     0.0763    0.8333    0.1399        12
     low_fire     0.8416    0.8397    0.8407       443
moderate_fire     0.0000    0.0000    0.0000        77
    high_fire     0.0000    0.0000    0.0000        41

     accuracy                         0.6667       573
    macro avg     0.2295    0.4183    0.2451       573
 weighted avg     0.6523    0.6667    0.6529       573

0.4182656132430399


In [ ]:
svm = SVC(C=2.5, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=2.5, class_weight='balanced', gamma='auto', kernel='linear',
    max_iter=1000)

{'fit_time': array([0.0208261 , 0.02381968]), 'score_time': array([0.00874829, 0.00929475]), 'test_score': array([0.66541353, 0.54425612])}
               precision    recall  f1-score   support

      no_fire     0.0763    0.8333    0.1399        12
     low_fire     0.8416    0.8397    0.8407       443
moderate_fire     0.0000    0.0000    0.0000        77
    high_fire     0.0000    0.0000    0.0000        41

     accuracy                         0.6667       573
    macro avg     0.2295    0.4183    0.2451       573
 weighted avg     0.6523    0.6667    0.6529       573

0.4182656132430399


In [ ]:
svm = SVC(C=10, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=10, class_weight='balanced', gamma='auto', kernel='linear', max_iter=1000)

{'fit_time': array([0.04172826, 0.05071115]), 'score_time': array([0.01503706, 0.02283096]), 'test_score': array([0.66917293, 0.51224105])}
               precision    recall  f1-score   support

      no_fire     0.0594    0.5000    0.1062        12
     low_fire     0.7361    0.1196    0.2058       443
moderate_fire     0.1557    0.3377    0.2131        77
    high_fire     0.0687    0.3902    0.1168        41

     accuracy                         0.1763       573
    macro avg     0.2550    0.3369    0.1605       573
 weighted avg     0.5962    0.1763    0.1983       573

0.3368862665716159


In [ ]:
svm = SVC(C=50, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=50, class_weight='balanced', gamma='auto', kernel='linear', max_iter=1000)

{'fit_time': array([0.08619833, 0.06272936]), 'score_time': array([0.042413  , 0.01412845]), 'test_score': array([0.15601504, 0.12429379])}
               precision    recall  f1-score   support

      no_fire     0.0256    0.2500    0.0465        12
     low_fire     0.8333    0.1919    0.3119       443
moderate_fire     0.2000    0.0779    0.1121        77
    high_fire     0.0710    0.5610    0.1260        41

     accuracy                         0.2042       573
    macro avg     0.2825    0.2702    0.1492       573
 weighted avg     0.6768    0.2042    0.2662       573

0.2701928192107403


In [ ]:
svm = SVC(C=100, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=100, class_weight='balanced', gamma='auto', kernel='linear',
    max_iter=1000)

{'fit_time': array([0.01682425, 0.02158284]), 'score_time': array([0.00671887, 0.00807405]), 'test_score': array([0.37781955, 0.28625235])}
               precision    recall  f1-score   support

      no_fire     0.0606    0.3333    0.1026        12
     low_fire     0.7273    0.0542    0.1008       443
moderate_fire     0.1263    0.4675    0.1989        77
    high_fire     0.0635    0.2927    0.1043        41

     accuracy                         0.1326       573
    macro avg     0.2444    0.2869    0.1267       573
 weighted avg     0.5851    0.1326    0.1143       573

0.28693119998245803


In [ ]:
svm = SVC(kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(class_weight='balanced', gamma='auto', kernel='linear', max_iter=1000)

{'fit_time': array([0.07338238, 0.07909799]), 'score_time': array([0.02931714, 0.03824329]), 'test_score': array([0.66541353, 0.54990584])}
               precision    recall  f1-score   support

      no_fire     0.0763    0.8333    0.1399        12
     low_fire     0.8416    0.8397    0.8407       443
moderate_fire     0.0000    0.0000    0.0000        77
    high_fire     0.0000    0.0000    0.0000        41

     accuracy                         0.6667       573
    macro avg     0.2295    0.4183    0.2451       573
 weighted avg     0.6523    0.6667    0.6529       573

0.4182656132430399


####Other parameter values

In [ ]:
svm = SVC(C=5, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=5, class_weight='balanced', gamma='auto', kernel='linear', max_iter=1000)

{'fit_time': array([0.02149367, 0.0261023 ]), 'score_time': array([0.00857949, 0.00977993]), 'test_score': array([0.66541353, 0.54613936])}
               precision    recall  f1-score   support

      no_fire     0.0763    0.8333    0.1399        12
     low_fire     0.8571    0.7178    0.7813       443
moderate_fire     0.0000    0.0000    0.0000        77
    high_fire     0.1127    0.1951    0.1429        41

     accuracy                         0.5864       573
    macro avg     0.2615    0.4366    0.2660       573
 weighted avg     0.6723    0.5864    0.6172       573

0.43657206041586377


In [ ]:
svm = SVC(C=25, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=25, class_weight='balanced', gamma='auto', kernel='linear', max_iter=1000)

{'fit_time': array([0.02516818, 0.02643824]), 'score_time': array([0.00847149, 0.00922799]), 'test_score': array([0.5093985 , 0.62711864])}
               precision    recall  f1-score   support

      no_fire     0.0250    0.1667    0.0435        12
     low_fire     0.8333    0.5418    0.6566       443
moderate_fire     0.2099    0.2208    0.2152        77
    high_fire     0.0726    0.2195    0.1091        41

     accuracy                         0.4677       573
    macro avg     0.2852    0.2872    0.2561       573
 weighted avg     0.6782    0.4677    0.5453       573

0.2871797012288671


In [ ]:
svm = SVC(C=5, kernel='linear', gamma='auto', class_weight='balanced', max_iter=1000)
svm.fit(X_train, Y_train)
Y_pred = svm.predict(X_test)
print(cross_validate(svm, X_train, Y_train, cv=2))
report = classification_report(Y_test, Y_pred, labels=[0,1,2,3], target_names=["no_fire", "low_fire", "moderate_fire","high_fire"], digits=4)
print(report)
print(metrics.balanced_accuracy_score(Y_test, Y_pred))

SVC(C=5, class_weight='balanced', gamma='auto', kernel='linear', max_iter=1000)

{'fit_time': array([0.02269363, 0.02685618]), 'score_time': array([0.00926733, 0.01013088]), 'test_score': array([0.66541353, 0.54613936])}
               precision    recall  f1-score   support

      no_fire     0.0763    0.8333    0.1399        12
     low_fire     0.8571    0.7178    0.7813       443
moderate_fire     0.0000    0.0000    0.0000        77
    high_fire     0.1127    0.1951    0.1429        41

     accuracy                         0.5864       573
    macro avg     0.2615    0.4366    0.2660       573
 weighted avg     0.6723    0.5864    0.6172       573

0.43657206041586377
